# Chinese Product Search Chatbot

This notebook implements a chatbot that can search Chinese product pages like AliExpress based on user queries, providing relevant product recommendations.

## Install Required Libraries

We need to install several packages for our chatbot:
- LangChain components for orchestration
- LangGraph for workflow management
- Web scraping tools
- LLM integration

In [ ]:
# Install required packages
!pip install langchain langchain_core langgraph
!pip install openai python-dotenv
!pip install requests beautifulsoup4 selenium
!pip install chromadb langsmith

## Set Up API Keys and Environment Variables

We need to set up our API keys for OpenAI and configure LangSmith for tracing (optional but helpful for debugging).

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set up API keys
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Optional: Set up LangSmith for tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chinese-product-search-chatbot"

# Verify keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found. Please add it to your .env file.")

## Define the Chat Model

We'll use OpenAI's GPT-4 model with a low temperature setting to ensure precise responses about products.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Initialize the chat model
chat_model = ChatOpenAI(
    model="gpt-4",
    temperature=0.1,  # Low temperature for more deterministic responses
)

# Test the model
response = chat_model.invoke("Say hello in Chinese")
print(response.content)

## Integrate Web Search Functionality

We'll create functions to search for products on AliExpress and other Chinese e-commerce platforms.

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from typing import List, Dict, Any, Optional

class ChineseProductSearcher:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        }
    
    def search_aliexpress(self, query: str, max_results: int = 5) -> List[Dict[str, Any]]:
        """
        Search AliExpress for products matching the query.
        This is a simplified implementation - in production, you might use their API or more robust scraping.
        """
        # Encode the query for URL
        encoded_query = query.replace(' ', '+')
        url = f"https://www.aliexpress.com/wholesale?SearchText={encoded_query}"
        
        print(f"Searching AliExpress for: {query}")
        
        try:
            # In a production environment, consider using:
            # 1. AliExpress API if available
            # 2. Selenium for full browser rendering
            # 3. A proxy service to avoid IP blocks
            
            # For demo purposes, we'll return mock data
            # In production, replace this with actual scraping code
            mock_results = self._get_mock_results(query, max_results)
            return mock_results
            
        except Exception as e:
            print(f"Error searching AliExpress: {e}")
            return []
    
    def _get_mock_results(self, query: str, count: int) -> List[Dict[str, Any]]:
        """Generate mock product results for demonstration purposes."""
        products = []
        query_words = query.lower().split()
        
        mock_products = [
            {
                "title": "Professional Pizza Oven for Commercial Use",
                "price": "$899.99",
                "rating": 4.8,
                "url": "https://www.aliexpress.com/item/12345.html",
                "image": "https://example.com/pizza_oven.jpg",
                "shipping": "Free Shipping",
                "orders": 250
            },
            {
                "title": "Pizza Dough Mixer 30L Commercial Grade",
                "price": "$349.99",
                "rating": 4.7,
                "url": "https://www.aliexpress.com/item/23456.html",
                "image": "https://example.com/dough_mixer.jpg",
                "shipping": "$50.00",
                "orders": 120
            },
            {
                "title": "Pizza Peel Set with Handle - Aluminum",
                "price": "$29.99",
                "rating": 4.5,
                "url": "https://www.aliexpress.com/item/34567.html",
                "image": "https://example.com/pizza_peel.jpg",
                "shipping": "$5.99",
                "orders": 500
            },
            {
                "title": "Commercial Pizza Preparation Table Refrigerated",
                "price": "$1299.99",
                "rating": 4.9,
                "url": "https://www.aliexpress.com/item/45678.html",
                "image": "https://example.com/prep_table.jpg",
                "shipping": "$100.00",
                "orders": 50
            },
            {
                "title": "Pizza Box Bundle - 100 boxes",
                "price": "$39.99",
                "rating": 4.6,
                "url": "https://www.aliexpress.com/item/56789.html",
                "image": "https://example.com/pizza_boxes.jpg",
                "shipping": "$12.99",
                "orders": 300
            },
            {
                "title": "Digital Kitchen Scale for Pizza Ingredients",
                "price": "$24.99",
                "rating": 4.7,
                "url": "https://www.aliexpress.com/item/67890.html",
                "image": "https://example.com/kitchen_scale.jpg",
                "shipping": "Free Shipping",
                "orders": 450
            },
            {
                "title": "Pizza Restaurant Furniture Set - Tables and Chairs",
                "price": "$899.99",
                "rating": 4.4,
                "url": "https://www.aliexpress.com/item/78901.html",
                "image": "https://example.com/restaurant_furniture.jpg",
                "shipping": "$150.00",
                "orders": 30
            }
        ]
        
        # Select relevant mock products based on the query
        for product in mock_products:
            product_relevant = any(word in product["title"].lower() for word in query_words)
            if product_relevant or not query_words:  # If no specific query words, include all
                products.append(product)
            
            if len(products) >= count:
                break
                
        # If we still need more products, add some generic ones
        while len(products) < count:
            idx = len(products)
            products.append({
                "title": f"Product related to {query} - Item {idx}",
                "price": f"${19.99 + idx * 10}",
                "rating": 4.0 + (idx % 10) / 10,
                "url": f"https://www.aliexpress.com/item/{10000 + idx}.html",
                "image": f"https://example.com/product_{idx}.jpg",
                "shipping": "Free Shipping" if idx % 2 == 0 else f"${4.99 + idx}",
                "orders": 50 + idx * 5
            })
        
        return products[:count]

# Initialize the product searcher
product_searcher = ChineseProductSearcher()

# Test the search function
test_results = product_searcher.search_aliexpress("pizza oven", max_results=3)
for i, product in enumerate(test_results, 1):
    print(f"{i}. {product['title']} - {product['price']}")

## Create a Prompt Template for Product Search

We'll design prompt templates that instruct our chatbot how to handle product search requests and provide recommendations.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# System prompt that defines the chatbot's behavior
system_prompt = """You are a helpful shopping assistant that specializes in finding products from Chinese e-commerce sites like AliExpress.
Your goal is to help users find the best products based on their needs, budget, and preferences.

When recommending products, consider:
1. Price and value for money
2. Shipping options and costs
3. Seller ratings and reputation
4. Product reviews and ratings
5. Number of orders (popularity)

For each product recommendation, provide:
- Product name and brief description
- Price
- Rating (if available)
- Link to the product
- Any notable features or specifications

Be honest about the limitations of shopping from Chinese e-commerce sites, such as:
- Shipping times can be long (often 2-4 weeks)
- Product quality can vary
- Return processes might be complicated

If the user asks about setting up a business or needs multiple items, group your recommendations by category.
"""

# Create the chat prompt template
chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    HumanMessage(content="{input}"),
    MessagesPlaceholder(variable_name="search_results"),
])

# Example of how the prompt would be used
prompt_example = chat_prompt.format_messages(
    chat_history=[
        HumanMessage(content="I'm looking to set up a small pizzeria. What equipment do I need?"),
        AIMessage(content="Setting up a pizzeria requires several key pieces of equipment. I can help you find options for each category. What's your budget range and which items are you most interested in finding first?")
    ],
    input="I need a pizza oven and dough mixer to start with. Budget is around $2000 total.",
    search_results=[
        AIMessage(content="I found these products for pizza ovens and dough mixers:\n\n" + 
                  "\n".join([f"- {product['title']}: {product['price']} | Rating: {product['rating']}/5 | {product['orders']} orders | {product['url']}" 
                             for product in test_results]))
    ]
)

# Print the formatted prompt example
for message in prompt_example:
    print(f"{message.type}: {message.content[:100]}...")

## Implement Conversation History Management

We'll create a simple memory system to maintain conversation history.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from typing import List, Dict, Any, Optional, Tuple

class ConversationManager:
    def __init__(self, max_history_length: int = 10):
        self.conversations: Dict[str, List] = {}
        self.max_history_length = max_history_length
    
    def add_message(self, conversation_id: str, message: Any) -> None:
        """Add a message to the conversation history."""
        if conversation_id not in self.conversations:
            self.conversations[conversation_id] = []
        
        self.conversations[conversation_id].append(message)
        
        # Trim history if it exceeds maximum length
        if len(self.conversations[conversation_id]) > self.max_history_length * 2:  # *2 because each turn has 2 messages
            # Remove oldest messages (keep most recent interactions)
            self.conversations[conversation_id] = self.conversations[conversation_id][-self.max_history_length*2:]
    
    def get_history(self, conversation_id: str) -> List:
        """Get the conversation history for a given ID."""
        return self.conversations.get(conversation_id, [])
    
    def clear_history(self, conversation_id: str) -> None:
        """Clear the conversation history for a given ID."""
        if conversation_id in self.conversations:
            self.conversations[conversation_id] = []
    
    def extract_product_needs(self, conversation_id: str) -> List[str]:
        """
        Analyze conversation history to extract product needs.
        This is a simplified version - in production you might use an LLM to extract this information.
        """
        history = self.get_history(conversation_id)
        product_needs = []
        
        # Simple keyword-based extraction
        keywords = ["need", "looking for", "find", "search", "buy", "purchase"]
        
        for message in history:
            if isinstance(message, HumanMessage):
                content = message.content.lower()
                for keyword in keywords:
                    if keyword in content:
                        # Extract the part after the keyword
                        start_idx = content.find(keyword) + len(keyword)
                        product_need = content[start_idx:].strip()
                        if product_need and len(product_need) > 3:  # Simple filtering
                            product_needs.append(product_need)
        
        return product_needs

# Initialize the conversation manager
conversation_manager = ConversationManager()

# Test conversation history
test_conversation_id = "test-user-123"
conversation_manager.add_message(test_conversation_id, HumanMessage(content="I need to find a pizza oven for my new restaurant."))
conversation_manager.add_message(test_conversation_id, AIMessage(content="I can help you find pizza ovens. What's your budget and size requirements?"))
conversation_manager.add_message(test_conversation_id, HumanMessage(content="Budget is around $1000 and I need something that can make 2 pizzas at once."))

# Get history
history = conversation_manager.get_history(test_conversation_id)
print(f"Conversation history has {len(history)} messages")
for i, msg in enumerate(history):
    print(f"{i+1}. {msg.type}: {msg.content[:50]}...")

# Extract product needs
needs = conversation_manager.extract_product_needs(test_conversation_id)
print("\nExtracted product needs:", needs)

## Build the Chatbot Workflow

Now we'll combine everything into a LangGraph workflow with memory management.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage

# Define the state for our graph
class ChainState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], "The messages in the conversation"]
    search_results: Annotated[List[Dict], "Results from product search"]
    user_needs: Annotated[List[str], "Extracted user needs"]
    conversation_id: str

# Define the nodes for our graph
def extract_search_query(state: ChainState) -> ChainState:
    """Extract search queries from the conversation."""
    # Get the most recent message
    last_message = state["messages"][-1]
    
    if not isinstance(last_message, HumanMessage):
        return state

    # Use an LLM to extract search queries
    # For simplicity, we'll use a rule-based approach here
    content = last_message.content.lower()
    
    # Check if the message contains search intent
    search_keywords = ["find", "search", "looking for", "need", "want", "buy", "purchase", "recommend"]
    has_search_intent = any(keyword in content for keyword in search_keywords)
    
    if has_search_intent:
        # Extract the query - in production, use an LLM for better extraction
        user_needs = []
        words = content.split()
        start_idx = 0
        
        for keyword in search_keywords:
            if keyword in content:
                if " ".join(keyword.split()) in content:  # Make sure we match the full keyword
                    keyword_idx = content.find(keyword)
                    if keyword_idx > -1:
                        start_idx = keyword_idx + len(keyword)
                        query = content[start_idx:].strip()
                        # Remove punctuation at the end
                        if query and query[-1] in ".!?":
                            query = query[:-1]
                        if query:
                            user_needs.append(query)
        
        if not user_needs:
            # Fallback: use the entire message as the query
            user_needs = [content]
        
        # Update the state
        state["user_needs"] = user_needs
    
    return state

def search_products(state: ChainState) -> ChainState:
    """Search for products based on extracted queries."""
    user_needs = state.get("user_needs", [])
    
    if not user_needs:
        # No search queries found
        return state
    
    # Perform product search for each need
    all_results = []
    for need in user_needs:
        results = product_searcher.search_aliexpress(need, max_results=3)
        all_results.extend(results)
    
    # Update the state with search results
    state["search_results"] = all_results
    
    # Add search results as a message for the LLM
    if all_results:
        results_text = "Based on your request, I found these products:\n\n"
        for i, product in enumerate(all_results, 1):
            results_text += (f"{i}. {product['title']}\n"
                            f"   Price: {product['price']}\n"
                            f"   Rating: {product['rating']}/5 ({product['orders']} orders)\n"
                            f"   Shipping: {product['shipping']}\n"
                            f"   Link: {product['url']}\n\n")
        
        search_message = AIMessage(content=results_text)
        state["messages"].append(search_message)
    
    return state

def generate_response(state: ChainState) -> ChainState:
    """Generate a response using the LLM."""
    # Extract chat history and the latest user message
    conversation_id = state["conversation_id"]
    chat_history = state["messages"][:-2] if len(state["messages"]) > 2 else []
    user_message = state["messages"][-1] if isinstance(state["messages"][-1], HumanMessage) else state["messages"][-2]
    
    # Prepare the prompt
    prompt = chat_prompt.format_messages(
        chat_history=chat_history,
        input=user_message.content,
        search_results=state["messages"][-1:] if "search_results" in state and state["search_results"] else []
    )
    
    # Generate a response using the LLM
    response = chat_model.invoke(prompt)
    
    # Add the response to the conversation
    state["messages"].append(response)
    
    # Store in conversation history
    conversation_manager.add_message(conversation_id, user_message)
    conversation_manager.add_message(conversation_id, response)
    
    return state

# Create the graph
workflow = StateGraph(ChainState)

# Add nodes
workflow.add_node("extract_query", extract_search_query)
workflow.add_node("search_products", search_products)
workflow.add_node("generate_response", generate_response)

# Define the edges
workflow.add_edge("extract_query", "search_products")
workflow.add_edge("search_products", "generate_response")
workflow.add_edge("generate_response", END)

# Set the entry point
workflow.set_entry_point("extract_query")

# Compile the graph
chain = workflow.compile()

## Test the Chatbot with Example Queries

Let's test our chatbot with some example queries to see how it performs.

In [ ]:
# Test function
def test_chatbot(conversation_id: str, user_message: str) -> None:
    """Test the chatbot with a user message."""
    print(f"User: {user_message}")
    
    # Get the conversation history
    history = conversation_manager.get_history(conversation_id)
    
    # Create the initial state
    initial_state = {
        "messages": history + [HumanMessage(content=user_message)],
        "search_results": [],
        "user_needs": [],
        "conversation_id": conversation_id
    }
    
    # Run the chain
    final_state = chain.invoke(initial_state)
    
    # Print the response
    response = final_state["messages"][-1]
    print(f"Chatbot: {response.content}\n")
    
    return final_state

# Test with a conversation
conversation_id = "test-conversation-123"

print("Test 1: Initial Query about Pizzeria Equipment")
test_chatbot(conversation_id, "What do I need to set up a pizzeria?")

print("Test 2: More Specific Query")
test_chatbot(conversation_id, "I need a good pizza oven that can fit at least 2 pizzas at once")

print("Test 3: Budget Constraint")
test_chatbot(conversation_id, "My budget for all equipment is around $3000. What would you recommend?")

print("Test 4: Non-product Query")
test_chatbot(conversation_id, "How long does shipping usually take from AliExpress?")

print("Test 5: Query about Reliability")
test_chatbot(conversation_id, "Are these pizza ovens reliable? I'm concerned about buying from overseas.")

## Conclusion and Next Steps

We've built a basic Chinese product search chatbot that:
1. Extracts search queries from user messages
2. Searches for products on Chinese e-commerce platforms
3. Generates helpful responses with product recommendations

To improve this chatbot, consider:
- Implementing more robust product search with real API integrations
- Adding filters for price, rating, shipping options, etc.
- Incorporating translation for Chinese product descriptions
- Building a user interface for the chatbot
- Adding image recognition to process product photos
- Implementing a recommendation system based on user preferences